# Multi-Session PCA with Train/Test Split

Testing the new train/test split functionality.

In [ ]:
# Imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import and reload modules
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session

import multi_session_pca
importlib.reload(multi_session_pca)
from multi_session_pca import MultiSessionPCA

print("✓ Imports loaded successfully!")

## Initialize with Configuration

In [ ]:
# Initialize with configuration
analyzer = MultiSessionPCA({
    'epok': [-100, 200],
    'bin_size': 1,
    'smooth_ker_size': 25,
    'alignment_point': 'go_cue',
    'ssd_number': 2,
    'normalize': False,
    'delta': False,
    'subtract_average_PSTH': False,
    'n_pca_components': 5
})

print(f"is_split flag: {analyzer.is_split}")

## Load and Validate Data

In [ ]:
# Load data
base_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
pickle_file = base_path / 'msn_fiona_cell_trial_data.pkl'

analyzer.load_data(pickle_file)

In [ ]:
# Validate sessions
analyzer.validate_all_sessions(verbose=False)
print(f"Valid sessions: {len(analyzer.valid_sessions)}")

## Extract PSTHs with Train/Test Split

**Key test**: Using `split_train_test=True` with 50/50 split and fixed random seed

In [ ]:
# Extract with train/test split
sessions_PSTHs = analyzer.extract_all_sessions_parallel(
    split_train_test=True,
    test_fraction=0.5,
    random_state=42
)

print(f"\nis_split flag after extraction: {analyzer.is_split}")

## Concatenate Sessions

Should create separate train and test matrices

In [ ]:
analyzer.concatenate_sessions()

## Verify Split Data

Check that we have both train and test matrices

In [ ]:
print("Train matrices:")
print(f"  GO Left Train: {analyzer.combined_go_left_train.shape if analyzer.combined_go_left_train is not None else 'None'}")
print(f"  GO Right Train: {analyzer.combined_go_right_train.shape if analyzer.combined_go_right_train is not None else 'None'}")
print(f"  STOP Left Train: {analyzer.combined_stop_left_train.shape if analyzer.combined_stop_left_train is not None else 'None'}")
print(f"  STOP Right Train: {analyzer.combined_stop_right_train.shape if analyzer.combined_stop_right_train is not None else 'None'}")

print("\nTest matrices:")
print(f"  GO Left Test: {analyzer.combined_go_left_test.shape if analyzer.combined_go_left_test is not None else 'None'}")
print(f"  GO Right Test: {analyzer.combined_go_right_test.shape if analyzer.combined_go_right_test is not None else 'None'}")
print(f"  STOP Left Test: {analyzer.combined_stop_left_test.shape if analyzer.combined_stop_left_test is not None else 'None'}")
print(f"  STOP Right Test: {analyzer.combined_stop_right_test.shape if analyzer.combined_stop_right_test is not None else 'None'}")

print(f"\nTotal cells: {analyzer.n_cells}")

## Fit PCA on TRAIN Data Only

In [ ]:
analyzer.fit_pca(pca_type='PCA', subtract_average=True)

## Project Both Train and Test Data

In [ ]:
analyzer.project_all_conditions()

## Verify Projections

Check that we have both train and test PC projections

In [ ]:
print("Train PC projections:")
print(f"  GO Left PCs: {analyzer.go_left_PCs.shape if analyzer.go_left_PCs is not None else 'None'}")
print(f"  GO Right PCs: {analyzer.go_right_PCs.shape if analyzer.go_right_PCs is not None else 'None'}")
print(f"  STOP Left PCs: {analyzer.stop_left_PCs.shape if analyzer.stop_left_PCs is not None else 'None'}")
print(f"  STOP Right PCs: {analyzer.stop_right_PCs.shape if analyzer.stop_right_PCs is not None else 'None'}")

print("\nTest PC projections:")
print(f"  GO Left PCs Test: {analyzer.go_left_PCs_test.shape if analyzer.go_left_PCs_test is not None else 'None'}")
print(f"  GO Right PCs Test: {analyzer.go_right_PCs_test.shape if analyzer.go_right_PCs_test is not None else 'None'}")
print(f"  STOP Left PCs Test: {analyzer.stop_left_PCs_test.shape if analyzer.stop_left_PCs_test is not None else 'None'}")
print(f"  STOP Right PCs Test: {analyzer.stop_right_PCs_test.shape if analyzer.stop_right_PCs_test is not None else 'None'}")

## Visualize Train Data (3D)

In [ ]:
# Plot 3D trajectory for TRAIN data
analyzer.plot_3d_trajectory()

## Visualize Test Data (3D)

Manually create a plot for test data to compare with train

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Create 3D plot for TEST data
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# GO trajectories (test)
ax.plot(analyzer.go_left_PCs_test[0, :], analyzer.go_left_PCs_test[1, :], analyzer.go_left_PCs_test[2, :],
        color='blue', linewidth=2, label='GO Left (180°)', alpha=0.8)
ax.plot(analyzer.go_right_PCs_test[0, :], analyzer.go_right_PCs_test[1, :], analyzer.go_right_PCs_test[2, :],
        color='green', linewidth=2, label='GO Right (0°)', alpha=0.8)

# STOP trajectories (test)
ax.plot(analyzer.stop_left_PCs_test[0, :], analyzer.stop_left_PCs_test[1, :], analyzer.stop_left_PCs_test[2, :],
        color='red', linewidth=2, label='STOP Left (180°)', linestyle='dashed', alpha=0.8)
ax.plot(analyzer.stop_right_PCs_test[0, :], analyzer.stop_right_PCs_test[1, :], analyzer.stop_right_PCs_test[2, :],
        color='orange', linewidth=2, label='STOP Right (0°)', linestyle='dashed', alpha=0.8)

# Start markers
ax.scatter(analyzer.go_left_PCs_test[0, 0], analyzer.go_left_PCs_test[1, 0], analyzer.go_left_PCs_test[2, 0],
          marker='^', s=200, color='blue', edgecolors='black', linewidths=2, zorder=5)
ax.scatter(analyzer.go_right_PCs_test[0, 0], analyzer.go_right_PCs_test[1, 0], analyzer.go_right_PCs_test[2, 0],
          marker='^', s=200, color='green', edgecolors='black', linewidths=2, zorder=5)
ax.scatter(analyzer.stop_left_PCs_test[0, 0], analyzer.stop_left_PCs_test[1, 0], analyzer.stop_left_PCs_test[2, 0],
          marker='*', s=200, color='red', edgecolors='black', linewidths=2, zorder=5)
ax.scatter(analyzer.stop_right_PCs_test[0, 0], analyzer.stop_right_PCs_test[1, 0], analyzer.stop_right_PCs_test[2, 0],
          marker='*', s=200, color='orange', edgecolors='black', linewidths=2, zorder=5)

ax.set_xlabel('PC1', fontsize=14)
ax.set_ylabel('PC2', fontsize=14)
ax.set_zlabel('PC3', fontsize=14)
ax.set_title(f'3D PC Trajectories - TEST DATA (n={len(analyzer.valid_sessions)} sessions, {analyzer.n_cells} cells)',
            fontsize=16)
ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

## Compare Train vs Test PC Trajectories

Plot PC1 timeseries for both train and test to see similarity

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Train PC1
ax = axes[0]
ax.plot(analyzer.time, analyzer.go_left_PCs[0, :], 'b-', lw=2, label='GO Left')
ax.plot(analyzer.time, analyzer.go_right_PCs[0, :], 'g-', lw=2, label='GO Right')
ax.plot(analyzer.time, analyzer.stop_left_PCs[0, :], 'r--', lw=2, label='STOP Left')
ax.plot(analyzer.time, analyzer.stop_right_PCs[0, :], '--', color='orange', lw=2, label='STOP Right')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax.set_ylabel('PC1')
ax.set_title('PC1 over Time - TRAIN')
ax.legend()
ax.grid(True, alpha=0.3)

# Test PC1
ax = axes[1]
ax.plot(analyzer.time, analyzer.go_left_PCs_test[0, :], 'b-', lw=2, label='GO Left')
ax.plot(analyzer.time, analyzer.go_right_PCs_test[0, :], 'g-', lw=2, label='GO Right')
ax.plot(analyzer.time, analyzer.stop_left_PCs_test[0, :], 'r--', lw=2, label='STOP Left')
ax.plot(analyzer.time, analyzer.stop_right_PCs_test[0, :], '--', color='orange', lw=2, label='STOP Right')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('PC1')
ax.set_title('PC1 over Time - TEST')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("If train/test split is working correctly, the trajectories should be similar but not identical.")

## Test Summary

Verify key functionality

In [ ]:
print("=" * 80)
print("TRAIN/TEST SPLIT FUNCTIONALITY TEST")
print("=" * 80)

print(f"\n✓ Split flag set: {analyzer.is_split}")
print(f"✓ Train matrices created: {analyzer.combined_go_left_train is not None}")
print(f"✓ Test matrices created: {analyzer.combined_go_left_test is not None}")
print(f"✓ PCA fitted on train data")
print(f"✓ Train projections created: {analyzer.go_left_PCs is not None}")
print(f"✓ Test projections created: {analyzer.go_left_PCs_test is not None}")

print(f"\nData shapes:")
print(f"  Train GO matrix: {analyzer.combined_go_left_train.shape}")
print(f"  Test GO matrix: {analyzer.combined_go_left_test.shape}")
print(f"  Train PC projection: {analyzer.go_left_PCs.shape}")
print(f"  Test PC projection: {analyzer.go_left_PCs_test.shape}")

print(f"\nPCA variance explained:")
for i, var in enumerate(analyzer.pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var*100:.2f}%")
print(f"  Total: {analyzer.pca.explained_variance_ratio_.sum()*100:.2f}%")

print("\n" + "=" * 80)
print("ALL TESTS PASSED!")
print("=" * 80)